# Reindex chunks Qdrant với FastEmbed BM25 trên Colab

Notebook này rebuild **chỉ Qdrant collection** cho chunks sau khi thêm `fastembed`:

- Xóa + tạo lại collection `history_vn_chunks` với dense vector mặc định và sparse vector named `bm25`.
- Embed `embedding_text` bằng `AITeamVN/Vietnamese_Embedding`.
- Encode BM25 bằng FastEmbed model `Qdrant/bm25`.
- Upsert point id = `uuid5(chunk_id)`, khớp `apps/agent-service/app/core/qdrant.py`.

Notebook **không đụng Postgres/Neo4j**. Trước khi chạy, upload `dataset/chunks_llm.json` lên Colab hoặc mount Google Drive rồi chỉnh `CHUNKS_FILE`.

## 1. Cài thư viện

In [2]:
!nvidia-smi

Wed Jul  1 03:46:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q "qdrant-client>=1.11,<2" sentence-transformers fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 95.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 36.2 MB/s eta 0:00:00


## 2. Config

Khuyến nghị đặt `QDRANT_HOST`, `QDRANT_API_KEY` trong Colab Secrets. Nếu không dùng Secrets thì điền trực tiếp vào biến bên dưới.

In [9]:
def _secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get(name) or default
    except Exception:
        return default


QDRANT_HOST = _secret("QDRANT_HOST", "114.29.239.135")
QDRANT_PORT = int(_secret("QDRANT_PORT", "6333"))
QDRANT_API_KEY = _secret("QDRANT_API_KEY", "doantotnghiep2026")
QDRANT_HTTPS = _secret("QDRANT_HTTPS", "false").lower() == "true"

COLLECTION = "history_vn_chunks"
CHUNKS_FILE = "D:\VFS\dataset\chunks_llm.json"
PROGRESS_FILE = "reindex_fastembed_progress.json"

EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding"
EMBEDDING_DIM = 1024
EMBEDDING_MAX_TOKENS = 2048

BM25_MODEL = "Qdrant/bm25"
SPARSE_VECTOR_NAME = "bm25"

# T4/L4 Colab thường ổn với default này. Nếu OOM, giảm EMBED_BATCH_SIZE xuống 8.
UPSERT_BATCH_SIZE = 64
EMBED_BATCH_SIZE = 16

# Reindex sạch cần xóa collection cũ. Để tránh bấm nhầm, phải đổi confirm string.
RECREATE_COLLECTION = True
CONFIRM_RESET = "REINDEX_QDRANT"  # đổi thành "REINDEX_QDRANT" trước khi chạy cell recreate

<>:16: SyntaxWarning: invalid escape sequence '\V'
<>:16: SyntaxWarning: invalid escape sequence '\V'
/tmp/ipykernel_9131/2070060518.py:16: SyntaxWarning: invalid escape sequence '\V'
  CHUNKS_FILE = "D:\VFS\dataset\chunks_llm.json"


## 3. Upload `chunks_llm.json` nếu chưa có

In [10]:
from pathlib import Path

if not Path(CHUNKS_FILE).exists():
    try:
        from google.colab import files  # type: ignore

        print(f"Chưa thấy {CHUNKS_FILE}. Hãy upload file dataset/chunks_llm.json từ repo.")
        files.upload()
    except Exception as exc:
        print(f"Không mở được uploader tự động: {exc}")

assert Path(CHUNKS_FILE).exists(), f"Không tìm thấy {CHUNKS_FILE}"

Chưa thấy D:\VFS\dataset\chunks_llm.json. Hãy upload file dataset/chunks_llm.json từ repo.


KeyboardInterrupt: 

## 4. Import + helper khớp code repo

In [ ]:
import gc
import json
import re
import uuid
from pathlib import Path

try:
    import torch
except Exception:
    torch = None

from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    Fusion,
    FusionQuery,
    Modifier,
    PointStruct,
    Prefetch,
    SparseVector,
    SparseVectorParams,
    VectorParams,
)
from sentence_transformers import SentenceTransformer

# Namespace cố định, phải khớp apps/agent-service/app/core/qdrant.py::_NS
_NS = uuid.UUID("a3f1c2e4-5b6d-4e8a-9c0f-1d2e3f4a5b6c")
_HEADING_KEY_RE = re.compile(r"^h(\d+)$", re.IGNORECASE)


def point_id_for(chunk_id: str) -> str:
    return str(uuid.uuid5(_NS, chunk_id))


def _string_list(value) -> list[str]:
    if not isinstance(value, list):
        return []
    return [str(item).strip() for item in value if str(item).strip()]


def build_heading_path(metadata: dict) -> list[str]:
    headings = metadata.get("headings")
    if not isinstance(headings, dict):
        return []

    ordered = []
    for key, value in headings.items():
        match = _HEADING_KEY_RE.match(str(key))
        text = str(value).strip() if value else ""
        if match and text:
            ordered.append((int(match.group(1)), text))
    return [text for _, text in sorted(ordered)]


def make_payload(chunk: dict) -> dict:
    metadata = chunk.get("metadata") or {}
    return {
        "chunk_id": chunk["chunk_id"],
        "heading_path": build_heading_path(metadata),
        "events": _string_list(metadata.get("events")),
        "actors": _string_list(metadata.get("actors")),
        "times": _string_list(metadata.get("times")),
        "locations": _string_list(metadata.get("locations")),
    }


def chunk_text_for_embedding(chunk: dict) -> str:
    return str(chunk.get("embedding_text") or chunk["text"])

## 5. Load và kiểm tra chunks

In [ ]:
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

assert isinstance(raw_chunks, list) and raw_chunks, "chunks_llm.json phải là JSON list không rỗng"

missing = [i for i, c in enumerate(raw_chunks) if not c.get("chunk_id") or not c.get("text")]
assert not missing, f"Có chunk thiếu chunk_id/text ở index: {missing[:10]}"

ids = [str(c["chunk_id"]) for c in raw_chunks]
dupes = sorted({cid for cid in ids if ids.count(cid) > 1})
assert not dupes, f"chunk_id bị trùng: {dupes[:10]}"

print(f"Loaded {len(raw_chunks):,} chunks")
print("Sample chunk_id:", raw_chunks[0]["chunk_id"])
print("Sample payload:", make_payload(raw_chunks[0]))

## 6. Kết nối Qdrant + recreate collection dense+sparse

In [ ]:
if not QDRANT_HOST or QDRANT_HOST == "YOUR_QDRANT_HOST":
    raise ValueError("Hãy điền QDRANT_HOST hoặc set Colab Secret QDRANT_HOST")

client = QdrantClient(
    host=QDRANT_HOST,
    port=QDRANT_PORT,
    api_key=QDRANT_API_KEY or None,
    https=QDRANT_HTTPS,
    timeout=120,
)

print("Connected:", client.get_collections())

if RECREATE_COLLECTION:
    if CONFIRM_RESET != "REINDEX_QDRANT":
        raise RuntimeError('Đổi CONFIRM_RESET thành "REINDEX_QDRANT" để xác nhận xóa collection cũ.')

    if client.collection_exists(COLLECTION):
        client.delete_collection(COLLECTION)
        print(f"Deleted old collection: {COLLECTION}")

    progress_path = Path(PROGRESS_FILE)
    if progress_path.exists():
        progress_path.unlink()
        print(f"Removed stale progress file: {PROGRESS_FILE}")

if not client.collection_exists(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
        sparse_vectors_config={
            SPARSE_VECTOR_NAME: SparseVectorParams(modifier=Modifier.IDF),
        },
    )
    print(f"Created collection: {COLLECTION}")
else:
    print(f"Collection already exists: {COLLECTION}")

info = client.get_collection(COLLECTION)
print(info)

## 7. Load dense model + FastEmbed BM25

In [ ]:
device = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
dense_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
dense_model.max_seq_length = EMBEDDING_MAX_TOKENS

sparse_model = SparseTextEmbedding(BM25_MODEL)

print(f"Dense model loaded on: {dense_model.device}")
print(f"Sparse model loaded: {BM25_MODEL}")

## 8. Embed dense + encode sparse + upsert

In [ ]:
def encode_sparse_documents(texts: list[str]) -> list[tuple[list[int], list[float]]]:
    embeddings = sparse_model.embed(texts)
    return [(e.indices.tolist(), e.values.tolist()) for e in embeddings]


def load_progress(path: str) -> set[str]:
    p = Path(path)
    if not p.exists():
        return set()
    with open(p, "r", encoding="utf-8") as f:
        return set(json.load(f))


def save_progress(path: str, progress: set[str]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sorted(progress), f, ensure_ascii=False)


progress = load_progress(PROGRESS_FILE)
todo = [chunk for chunk in raw_chunks if chunk["chunk_id"] not in progress]
print(f"Already done: {len(progress):,}")
print(f"Need upsert: {len(todo):,}")

for start in range(0, len(todo), UPSERT_BATCH_SIZE):
    batch = todo[start : start + UPSERT_BATCH_SIZE]
    texts = [chunk_text_for_embedding(chunk) for chunk in batch]

    dense_vectors = dense_model.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    sparse_vectors = encode_sparse_documents(texts)

    points = [
        PointStruct(
            id=point_id_for(chunk["chunk_id"]),
            vector={
                "": dense_vec.tolist(),
                SPARSE_VECTOR_NAME: SparseVector(indices=sparse_idx, values=sparse_val),
            },
            payload=make_payload(chunk),
        )
        for chunk, dense_vec, (sparse_idx, sparse_val) in zip(batch, dense_vectors, sparse_vectors)
    ]

    client.upsert(collection_name=COLLECTION, points=points, wait=True)

    for chunk in batch:
        progress.add(chunk["chunk_id"])
    save_progress(PROGRESS_FILE, progress)

    done = len(progress)
    pct = done / len(raw_chunks) * 100
    print(f"[{done:,}/{len(raw_chunks):,} - {pct:.1f}%] batch {(start // UPSERT_BATCH_SIZE) + 1}")

    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Done reindex Qdrant.")

## 9. Verify collection và sample vector

In [ ]:
info = client.get_collection(COLLECTION)
print(f"Collection points: {info.points_count:,}")
print(f"Input chunks:       {len(raw_chunks):,}")
assert info.points_count == len(raw_chunks), "Point count không khớp chunks_llm.json"

points, _ = client.scroll(
    collection_name=COLLECTION,
    limit=1,
    with_payload=True,
    with_vectors=True,
)
assert points, "Không scroll được sample point"

sample = points[0]
print("Sample point id:", sample.id)
print("Sample chunk_id:", sample.payload.get("chunk_id"))
print("Vector keys:", list(sample.vector.keys()) if isinstance(sample.vector, dict) else type(sample.vector))

assert isinstance(sample.vector, dict), "Qdrant không trả vector dạng dict; kiểm tra schema collection"
assert "" in sample.vector, "Thiếu dense vector mặc định key rỗng"
assert SPARSE_VECTOR_NAME in sample.vector, f"Thiếu sparse vector {SPARSE_VECTOR_NAME}"
print("OK - dense + sparse vector đều có trên point.")

## 10. Smoke test query dense+sparse RRF

In [ ]:
query = "Phan Bội Châu và phong trào Đông Du"

dense_query = dense_model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)[0].tolist()

sparse_query = next(iter(sparse_model.query_embed(query)))
sparse_query = SparseVector(
    indices=sparse_query.indices.tolist(),
    values=sparse_query.values.tolist(),
)

result = client.query_points(
    collection_name=COLLECTION,
    prefetch=[
        Prefetch(query=dense_query, limit=5),
        Prefetch(query=sparse_query, using=SPARSE_VECTOR_NAME, limit=5),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
    with_payload=True,
)

for rank, point in enumerate(result.points, start=1):
    payload = point.payload or {}
    print(rank, point.score, payload.get("chunk_id"), payload.get("heading_path"))